In [1]:
# import modules
import os
import subprocess
import argparse
import pandas as pd
import logging
from scipy import stats
import drugsets_func as df_func


In [2]:
# TO DO: build in gene ID mapping to the .preds file (i.e., replace ENSGID w/ entrez depending on the .raw input file)
def normalize_pops_and_adjust_genes_out(magma_genes_out, pops_preds, adj_pops, join_str):
    try:
        # pops_df = pd.read_csv(pops_preds, sep='\t')
        pops_df = pops_preds
        if adj_pops == 'yes':
            logging.info("Using adjusted PoPS scores.")
            pops_df['normed_pops'] = stats.zscore(pops_df['PoPS_Z'])
        else:
            logging.info("Using unadjusted PoPS scores.")
            pops_df['normed_pops'] = stats.zscore(pops_df['PoPS_Score'])
        # compute p-values & merge with MAGMA genes.out
        pops_df['pvalue'] = 2 * stats.norm.sf(abs(pops_df['normed_pops']))
        # magma_df = pd.read_csv(magma_genes_out, delimiter= "\s+", comment='#')
        magma_df = magma_genes_out
        merged_df = magma_df.merge(pops_df[['ENSGID', 'normed_pops', 'pvalue']], left_on='GENE', right_on='ENSGID', how=join_str)
        # merged_df["normed_pops"] = merged_df["normed_pops"].fillna(merged_df["ZSTAT"])
        # merged_df["pvalue"] = merged_df["pvalue"].fillna(merged_df["P"])
        # merged_df.drop(columns=['ENSGID', 'ZSTAT', 'P'], inplace=True)
        merged_df.rename(columns={'normed_pops': 'ZSTAT', 'pvalue': 'P'}, inplace=True)
        # adjusted_genes_out = os.path.join(self.out_dir, "adjusted.genes.out")
        # merged_df.to_csv(adjusted_genes_out, sep='\t', index=False)
        return merged_df
    except Exception as e:
        logging.error(f"Error processing PoPs data: {e}")
        raise

In [3]:
def filter_genes(df, keep_genes):
    # Step 1: Identify genes present in the DataFrame but not in keep_genes
    all_genes = set(df["GENE"])
    keep_genes_set = set(keep_genes)
    missing_genes = all_genes - keep_genes_set

    # Step 2: Remove missing genes from each COVARS dict
    df["COVARS"] = df["COVARS"].apply(
        lambda covar_dict: {gene: val for gene, val in covar_dict.items() if gene not in missing_genes}
    )

    # Step 3: Remove rows corresponding to missing genes
    df_filtered = df[df["GENE"].isin(keep_genes)].reset_index(drop=True)

    return df_filtered

In [4]:
def save_triangular_raw(df, output_path):
    with open(output_path, 'w') as f:
        for i, row in df.iterrows():
            # Extract the first 9 fields
            base_cols = [str(row[col]) for col in df.columns[:9]]
            
            # Extract the covariance values as list
            covars = row['COVARS']
            covar_values = [str(covars[gene]) for gene in df.loc[:i-1, "GENE"] if gene in covars]
            
            # Combine and write as a space-separated line
            line = " ".join(base_cols + covar_values)
            f.write(line + "\n")


In [5]:
# load data
pops = pd.read_csv('~/phd_code/EXT_gwas_2025/data/Ext_2_AFR_EUR_PoPS.preds', sep='\t')
pops_genes = pops["ENSGID"].to_list()
print(len(set(pops_genes)))
pops.head()

18383


,ENSGID,PoPS_Score,Y,Y_proj,project_out_covariates_gene,feature_selection_gene,training_gene
0,ENSG00000186092,-0.243761,NaN,NaN,False,False,False
1,ENSG00000235249,-0.214060,NaN,NaN,False,False,False
2,ENSG00000185097,-0.252811,NaN,NaN,False,False,False
3,ENSG00000187634,-0.109047,-1.03700,-2.552717,True,True,True
4,ENSG00000188976,0.123352,0.15848,-1.688460,True,True,True


In [6]:
# load data
magma_out = pd.read_csv('~/phd_code/EXT_gwas_2025/data/magma_meta_genes_Ext_2_AFR_EUR.genes.out', sep='\s+', comment='#')
magma_out.head()

,GENE,CHR,START,STOP,NSNPS,NPARAM,N,DATASETS,ZSTAT,P
0,ENSG00000187634,1,860260,879955,35,12,285713,1,-1.03700,0.85014
1,ENSG00000268179,1,861264,866445,14,5,285713,1,-0.57691,0.71800
2,ENSG00000188976,1,879584,894689,53,14,285713,1,0.15848,0.43704
3,ENSG00000187961,1,895967,901095,11,6,285713,1,-0.94570,0.82785
4,ENSG00000187583,1,901877,911245,19,12,285713,1,-1.64760,0.95028


In [61]:
filepath = "/Users/nyb/phd_code/EXT_gwas_2025/data/magma_meta_raw_Ext_2_AFR_EUR.genes.raw"

# Initialize storage
data = []

# Open and parse manually
with open(filepath, 'r') as f:
    for line in f:
        # Skip comments
        if line.startswith("#"):
            continue

        # Split line into tokens
        tokens = line.strip().split()
        
        # First 9 columns go as-is
        fixed_cols = tokens[:9]
        
        # Remaining values are covariances
        covars = list(map(float, tokens[9:])) if len(tokens) > 9 else []
        
        # Append to data list
        data.append(fixed_cols + [covars])

# Define column names: first 9 + 1 for the covariance list
column_names = ['GENE', 'CHR', 'START', 'STOP', 'NSNPS', 'NPARAM', 'N', 'ZSTAT', 'P', 'COVARS']

# Create DataFrame
df = pd.DataFrame(data, columns=column_names)

# Optionally convert numeric columns
for col in ['CHR', 'START', 'STOP', 'NSNPS', 'NPARAM', 'N', 'ZSTAT', 'P']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [62]:
df.head()

,GENE,CHR,START,STOP,NSNPS,NPARAM,N,ZSTAT,P,COVARS
0,ENSG00000187634,1,860260,879955,35,12,285713,51.9143,-1.037040,[]
1,ENSG00000268179,1,861264,866445,14,5,285713,36.0000,-0.576898,[0.807556]
2,ENSG00000188976,1,879584,894689,53,14,285713,84.1698,0.158481,"[0.512363, 0.227097]"
3,ENSG00000187961,1,895967,901095,11,6,285713,57.0909,-0.945722,"[0.371712, 0.31706, 0.436713]"
4,ENSG00000187583,1,901877,911245,19,12,285713,48.0000,-1.647600,"[0.322912, 0.226417, 0.346092, 0.502276]"


In [63]:
# Initialize a new column to hold the mapped dictionaries
covar_dicts = []

# Loop through each row
for i in range(len(df)):
    # Get previous gene names up to current row (non-inclusive)
    prev_genes = df.loc[:i-1, "GENE"].tolist()
    
    # Get the current row's list of covariance values
    covars = df.at[i, "COVARS"]
    
    # Map previous genes to covariance values
    covar_dict = dict(zip(prev_genes, covars))
    
    # Append to list
    covar_dicts.append(covar_dict)

# Replace the COVARS column with the dictionary form
df["COVARS"] = covar_dicts

In [64]:
df.head()

,GENE,CHR,START,STOP,NSNPS,NPARAM,N,ZSTAT,P,COVARS
0,ENSG00000187634,1,860260,879955,35,12,285713,51.9143,-1.037040,{}
1,ENSG00000268179,1,861264,866445,14,5,285713,36.0000,-0.576898,{'ENSG00000187634': 0.807556}
2,ENSG00000188976,1,879584,894689,53,14,285713,84.1698,0.158481,"{'ENSG00000187634': 0.512363, 'ENSG00000268179..."
3,ENSG00000187961,1,895967,901095,11,6,285713,57.0909,-0.945722,"{'ENSG00000187634': 0.371712, 'ENSG00000268179..."
4,ENSG00000187583,1,901877,911245,19,12,285713,48.0000,-1.647600,"{'ENSG00000187634': 0.322912, 'ENSG00000268179..."


In [68]:
df_filtered = filter_genes(df, pops_genes)
df_filtered.head()

,GENE,CHR,START,STOP,NSNPS,NPARAM,N,ZSTAT,P,COVARS
0,ENSG00000187634,1,860260,879955,35,12,285713,51.9143,-1.037040,{}
1,ENSG00000188976,1,879584,894689,53,14,285713,84.1698,0.158481,{'ENSG00000187634': 0.512363}
2,ENSG00000187961,1,895967,901095,11,6,285713,57.0909,-0.945722,"{'ENSG00000187634': 0.371712, 'ENSG00000188976..."
3,ENSG00000187583,1,901877,911245,19,12,285713,48.0000,-1.647600,"{'ENSG00000187634': 0.322912, 'ENSG00000188976..."
4,ENSG00000187642,1,910579,917497,6,3,4203951,222.3120,-0.750306,"{'ENSG00000187634': 0.114707, 'ENSG00000188976..."


In [7]:
merged_df = magma_out.merge(pops[['ENSGID']], left_on='GENE', right_on='ENSGID', how="inner")
merged_df.drop(columns=['ENSGID'], inplace=True)
merged_df

,GENE,CHR,START,STOP,NSNPS,NPARAM,N,DATASETS,ZSTAT,P
0,ENSG00000187634,1,860260,879955,35,12,285713,1,-1.03700,0.850140
1,ENSG00000188976,1,879584,894689,53,14,285713,1,0.15848,0.437040
2,ENSG00000187961,1,895967,901095,11,6,285713,1,-0.94570,0.827850
3,ENSG00000187583,1,901877,911245,19,12,285713,1,-1.64760,0.950280
4,ENSG00000187642,1,910579,917497,6,3,4203951,2,-0.75031,0.773470
...,...,...,...,...,...,...,...,...,...,...
17878,ENSG00000100288,22,51017378,51039884,88,21,4203951,2,2.02120,0.021631
17879,ENSG00000008735,22,51039114,51052409,51,12,4203951,2,2.62610,0.004318
17880,ENSG00000100299,22,51061182,51066607,21,7,4203951,2,1.59420,0.055447
17881,ENSG00000251322,22,51112843,51171726,76,17,4203951,2,-0.43968,0.669910


In [8]:
merged_df.to_csv("~/Desktop/test.adjusted.genes.out", sep='\t', index=False)

In [ ]:
merged = normalize_pops_and_adjust_genes_out(magma_out, pops, 'no')
merged.head()

In [73]:
merged.to_csv("~/Desktop/test.adjusted.genes.out", sep='\t', index=False)